# Life Expectancy Prediction Analysis using Random Forest

This notebook performs regression analysis on global development indicators from the year 2022. The objective is to train a supervised machine learning regressor to predict a country's average **Life Expectancy at Birth** (`life_expectancy_at_birth`) based on a variety of socioeconomic and infrastructure indicators.

The analysis includes:
- Data loading and preparation
- Splitting data into training and test sets
- Training a Random Forest Regressor
- Evaluating the regressor using standard metrics ($R^2$, MAE, RMSE)
- Analyzing Feature Importance to identify the strongest predictors of longevity
- Validating the model using 5-Fold Cross-Validation
- Tuning hyperparameters using Grid Search (`GridSearchCV`)
- Benchmarking and comparing performance against regularized linear models (Lasso & Ridge) including linear coefficient impact analysis

## Importing libraries

In [ ]:
# 1. Importing the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [ ]:
# Visual configuration for plots
sns.set_theme(style="whitegrid")

## Reading and loading the dataset

We load the dataset containing the country socioeconomic indicators along with the K-Means cluster labels.

In [ ]:
# 2. Loading the data
df = pd.read_csv('../WDI-2022-clustering/WDI2022_clusters.csv')

# Definition of the target column
target_col = 'life_expectancy_at_birth'

# Removing rows where our target variable is empty
df = df.dropna(subset=[target_col])

In [ ]:
df.columns

### 3.1. Rationale for Excluding Child Mortality

The indicator `child_mortality_rate` (Child Mortality Rate) is explicitly dropped from the predictor features.

**Technical Rationale:** Child mortality shares an extremely strong and direct biological and demographic correlation with life expectancy at birth. Retaining it would make the regression task trivial, causing the model to bypass structural drivers. By excluding it, we force the model to predict life expectancy using macroeconomic, social, and health-system infrastructure features (such as GDP per capita, electricity access, health expenditure, and fertility rate).

## Data preparation and feature selection

We drop non-predictive columns (`Country Name`, `Country Code`, `Region`), alternative cluster labels, and the target variable itself.

In [ ]:
# 3. Separating predictor variables (X) and target variable (y)
cols_to_drop = ['Country Name', 'Country Code', 'Region', 'child_mortality_rate',  
                'Cluster_KMeans', 'Cluster_HDBSCAN', target_col] 
X = df.drop(columns=[col for col in cols_to_drop if col in df.columns])
y = df[target_col]

## Train-Test Split

We split the dataset into 80% for training and 20% for testing to evaluate the generalization capability of the models.

In [ ]:
# 4. Splitting into Train (80%) and Test (20%) datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Model Training

We train a Random Forest Regressor using 100 decision trees to predict the country's average life expectancy.

In [ ]:
# 5. Training the Regression Model
print("Training the Random Forest Regressor model...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

## Model Predictions

We use the trained model to make predictions on the unseen test dataset.

In [ ]:
# 6. Making predictions on test data
y_pred = rf_model.predict(X_test)

## Model Evaluation

We evaluate the model's performance on the test set using R² (explained variance), Mean Absolute Error (MAE), and Root Mean Squared Error (RMSE).

In [ ]:
# 7. Evaluating the Model
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\n--- Evaluation Metrics ---")
print(f"R² (Explanatory power): {r2:.4f} (Closer to 1.0 is better)")
print(f"MAE (Mean Absolute Error): {mae:.2f} years")
print(f"RMSE (Root Mean Squared Error): {rmse:.2f} years")

## 5-Fold Cross-Validation

To obtain a robust and unbiased estimation of our regressor's performance, we apply 5-Fold Cross-Validation to evaluate the average $R^2$, MAE, and RMSE.

In [ ]:
# 8. 5-Fold Cross-Validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = cross_val_score(rf_model, X, y, cv=cv, scoring='r2')
mae_scores = -cross_val_score(rf_model, X, y, cv=cv, scoring='neg_mean_absolute_error')
rmse_scores = np.sqrt(-cross_val_score(rf_model, X, y, cv=cv, scoring='neg_mean_squared_error'))

print("--- 5-Fold Cross-Validation Results ---")
print(f"Mean R²: {r2_scores.mean():.4f} (Standard Deviation: {r2_scores.std():.4f})")
print(f"Mean MAE: {mae_scores.mean():.2f} years (Standard Deviation: {mae_scores.std():.2f})")
print(f"Mean RMSE: {rmse_scores.mean():.2f} years (Standard Deviation: {rmse_scores.std():.2f})")

## Feature Importance

By extracting feature importances from the Random Forest model, we can identify which socioeconomic indicators are the strongest predictors of country-level life expectancy.

In [ ]:
# 9. Feature Importance Analysis
rf_feature_importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=rf_feature_importances.values, y=rf_feature_importances.index, hue=rf_feature_importances.index, palette="mako", legend=False)
plt.title("Socioeconomic Indicators Importance in Predicting Life Expectancy", fontsize=14)
plt.xlabel("Importance Score (MDI)")
plt.ylabel("Socioeconomic Indicators")
plt.tight_layout()
plt.show()

## Hyperparameter Tuning

We use `GridSearchCV` to optimize the Random Forest hyperparameters (such as maximum depth, split criteria, and estimators) to improve predictive accuracy.

In [ ]:
# 10. Hyperparameter Tuning using GridSearchCV
param_grid_reg = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'max_features': [1.0, 'sqrt', 'log2']
}

grid_search_reg = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid_reg,
    cv=cv,
    scoring='r2',
    n_jobs=1
)

print("Running Hyperparameter Search for Regression...")
grid_search_reg.fit(X_train, y_train)

print(f"Best parameters found: {grid_search_reg.best_params_}")

# Evaluating the optimized model on the test set
best_regressor = grid_search_reg.best_estimator_
best_pred_reg = best_regressor.predict(X_test)

best_r2 = r2_score(y_test, best_pred_reg)
best_mae = mean_absolute_error(y_test, best_pred_reg)
best_rmse = np.sqrt(mean_squared_error(y_test, best_pred_reg))

print("\n--- Optimized Regressor Test Set Results ---")
print(f"R²: {best_r2:.4f}")
print(f"MAE: {best_mae:.2f} years")
print(f"RMSE: {best_rmse:.2f} years")

## Comparison with Regularized Linear Models (Lasso & Ridge)

To evaluate the trade-off between complexity and interpretability, we compare the Random Forest results against **Lasso** and **Ridge** regression. We set up robust pipelines using `StandardScaler` to ensure all features are on the same scale for the linear models.

In [ ]:
# 11. Creating and Evaluating Pipelines for Lasso and Ridge Regression
lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', Lasso(alpha=0.1, random_state=42))
])

ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0, random_state=42))
])

# Fit and evaluate Lasso
lasso_pipeline.fit(X_train, y_train)
lasso_pred = lasso_pipeline.predict(X_test)
lasso_r2 = r2_score(y_test, lasso_pred)
lasso_mae = mean_absolute_error(y_test, lasso_pred)

# Fit and evaluate Ridge
ridge_pipeline.fit(X_train, y_train)
ridge_pred = ridge_pipeline.predict(X_test)
ridge_r2 = r2_score(y_test, ridge_pred)
ridge_mae = mean_absolute_error(y_test, ridge_pred)

print("--- Model Performance Comparison ---")
print(f"Lasso Regression: R² = {lasso_r2:.4f}, MAE = {lasso_mae:.2f} years")
print(f"Ridge Regression: R² = {ridge_r2:.4f}, MAE = {ridge_mae:.2f} years")
print(f"Random Forest (Baseline): R² = {r2:.4f}, MAE = {mae:.2f} years")
print(f"Random Forest (Optimized): R² = {best_r2:.4f}, MAE = {best_mae:.2f} years")

# Extracting Lasso coefficients for linear impact analysis
lasso_coefs = pd.Series(lasso_pipeline.named_steps['lasso'].coef_, index=X.columns).sort_values()

plt.figure(figsize=(10, 6))
sns.barplot(x=lasso_coefs.values, y=lasso_coefs.index, hue=lasso_coefs.index, palette="coolwarm", legend=False)
plt.title("Lasso Regression Coefficients (Life Expectancy Impact Analysis)", fontsize=14)
plt.xlabel("Linear Coefficient (Impact Size)")
plt.ylabel("Socioeconomic Indicators")
plt.axvline(0, color='red', linestyle='--')
plt.tight_layout()
plt.show()